In [ ]:
#-------------------------------------------------------------------------------
# Name:        02_create_input.data
# Purpose:     Create missing polygon input data for modelling
#
# Author:      Gijs van den Dool & Meggison Oritsemisan
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/1Xjhe6JXYHi8U_q-eseQ1nx4FUVBc-1Tk


In [ ]:
import os
import geemap
import json
from shapely.geometry import Point,LineString, Polygon

import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Local download of the labels prepared by the team and root of this notebook:
# change the root string to reflect your settings:

project_root = "/content/drive/MyDrive/Project Canopy Official Folder/Task 02 Data Preprocessing/Working Files/Experiment_IA" # Change Project root
task_folder = "01_Input_Data"
file_name = "Iter4_70_IA.geojson" # might be different for other areas

file_path = os.path.join(project_root, task_folder, file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
    print("File found")
else:
    print("File not found")


File found


In [ ]:
gdf_ISL_lines = gpd.read_file(file_path)

print(len(gdf_ISL_lines)) # total lines in the AoI
print(gdf_ISL_lines.crs)  # current projection
gdf_ISL_lines.head(3)

1
EPSG:4326


,Shape_Leng,Shape_Area,url,img_date,min_date,max_date,color,geometry
0,0.713247,0.003667,https://apps.sentinel-hub.com/eo-browser/?zoom...,2024-01-24,2023-12-31,2024-01-24,#d00000,"MULTIPOLYGON Z (((8.90144 4.94081 0.00000, 8.8..."


In [ ]:
def set_projection(gdf_input, org_CRS, new_CRS):
  # working on the projection system (CRS)
  if org_CRS != new_CRS:  # setting the new projection, buffers will be
                              # reprojected to the org_CRS
    gdf_output = gdf_input.to_crs(new_CRS)
  else:
    gdf_output = gdf_input

  print(gdf_output.crs)

  return gdf_output

In [ ]:
buffer_distance = 30 # in meters
AoI_id = "IA_70"   # will be different for other areas

# setting the projection
org_CRS = gdf_ISL_lines.crs
new_CRS = "EPSG:32632"       # this PROJCRS "WGS 84 / UTM zone 32N" in meters
gdf_ISL_buffer = set_projection(gdf_ISL_lines, org_CRS, new_CRS)

# Remove the columns that are not needed
gdf_ISL_buffer['AoI_id'] = AoI_id
gdf_ISL_buffer = gdf_ISL_buffer[['AoI_id', 'geometry']]

gdf_ISL_buffer

EPSG:32632


,AoI_id,geometry
0,IA_70,"MULTIPOLYGON Z (((489073.662 546121.788 0.000,..."


In [ ]:
# reset the projection system and save the buffer:
gdf_ISL_buffer = gdf_ISL_buffer.set_crs(new_CRS) #
gdf_ISL_buffer = gdf_ISL_buffer.to_crs(org_CRS)

# save the buffer
task_folder = "02_WorkingFiles"
working_folder = os.path.join(project_root, task_folder)
file_name = f"buffer_{AoI_id}.geojson" # might be different for other areas

file_path = os.path.join(working_folder, file_name)
gdf_ISL_buffer.to_file(file_path)


In [ ]:
gdf_ISL_buffer.explore()

In [ ]:
#-------------------------------------------------------------------------------
# With the buffers created we can go to the next step: creating a grid based
# on the AoI

In [ ]:
def create_grid(AoI_polygon, grid_distance, AoI_id):
#-------------------------------------------------------------------------------
# Name:        create_grid
# Purpose:     Create a grid of polygons inside the AoI based on a grid distance,
#              the grid distance is based on the patch size, and preset to be
#              128x128px, or 1280x1280m
# Returns:     GeoPandas Dataframe
#-------------------------------------------------------------------------------
  minx, miny, maxx, maxy = AoI_polygon.total_bounds

  width = grid_distance
  height = grid_distance
  rows = int(np.ceil((maxy - miny) /  height))
  cols = int(np.ceil((maxx - minx) / width))
  leftOriginX = int(minx)
  rightOriginX = int(minx) + width
  topOriginY = int(maxy)
  bottomOriginY = int(maxy)- height

  polygons = []
  grid_id_list = []
  for i in range(cols):
    topY = topOriginY
    bottomY =bottomOriginY
    for j in range(rows):
      polygons.append(Polygon([(leftOriginX, topY), (rightOriginX, topY),
                              (rightOriginX, bottomY), (leftOriginX, bottomY)]))
      grid_id = f"GID_{AoI_id}_{i}_{j}"
      grid_id_list.append(grid_id)
      topY = topY - height
      bottomY = bottomY - height

    # end for
    leftOriginX = leftOriginX + width
    rightOriginX = rightOriginX + width
  # end for

  grid_polygon = gpd.GeoDataFrame({'geometry':polygons})
  grid_polygon["GID"]=grid_id_list


  return grid_polygon

In [ ]:
working_folder = "01_Input_Data"
file_name = "IA_70.geojson" # might be different for other areas

# get grid file path
grid_path = os.path.join(project_root, working_folder, file_name)

# Test if the path is accessible
os.path.isfile(grid_path)

True

In [ ]:
gdf_AoI_id = gpd.read_file(grid_path)

print(len(gdf_AoI_id)) # total lines in the AoI
print(gdf_AoI_id.crs)  # current projection
gdf_AoI_id.head()

1
EPSG:4326


,image_AoI,min_date,max_date,url,image_date,geometry
0,IA_70,2023-12-31,2024-01-24,https://apps.sentinel-hub.com/eo-browser/?zoom...,2024-01-24,"POLYGON ((8.84439 4.92207, 8.91228 4.92332, 8...."


In [ ]:
# setting the projection
org_CRS = gdf_AoI_id.crs
new_CRS = "EPSG:32632"       # this PROJCRS "WGS 84 / UTM zone 32N" in meters
gdf_AoI_id = set_projection(gdf_AoI_id, org_CRS, new_CRS)

EPSG:32632


In [ ]:
# create the grid: IA
grid_distance = 2560 # 256px size at 10m ground resolution
gdf_grid = create_grid(gdf_AoI_id, grid_distance, AoI_id)

# reset the projection:
new_CRS = "EPSG:32632"
gdf_grid = gdf_grid.set_crs(new_CRS)
gdf_grid.to_crs(org_CRS, inplace=True)

task_folder = "02_WorkingFiles"
working_folder = os.path.join(project_root, task_folder)
file_name = f"grid_{AoI_id}.geojson" # might be different for other areas

file_path = os.path.join(working_folder, file_name)
gdf_grid.to_file(file_path)

In [ ]:
gdf_grid.head(5)

,geometry,GID
0,"POLYGON ((8.86580 4.99090, 8.84311 4.99090, 8....",GID_IA_70_0_0
1,"POLYGON ((8.86580 4.99090, 8.86580 4.96774, 8....",GID_IA_70_0_1
2,"POLYGON ((8.86580 4.96774, 8.86581 4.94458, 8....",GID_IA_70_0_2
3,"POLYGON ((8.86581 4.94458, 8.86581 4.92247, 8....",GID_IA_70_0_3
4,"POLYGON ((8.88889 4.99091, 8.86580 4.99090, 8....",GID_IA_70_1_0


In [ ]:
print(gdf_grid.crs)
print(gdf_ISL_buffer.crs)

gdf_mask_full = gpd.overlay(gdf_ISL_buffer, gdf_grid, how='union')
gdf_mask_full.head()

EPSG:4326
EPSG:4326


,AoI_id,GID,geometry
0,IA_70,GID_IA_70_0_0,"POLYGON Z ((8.86017 4.99525 0.00000, 8.86022 4..."
1,IA_70,GID_IA_70_0_1,"POLYGON Z ((8.85722 4.97485 0.00000, 8.85705 4..."
2,IA_70,GID_IA_70_0_2,"POLYGON Z ((8.85784 4.95559 0.00000, 8.85440 4..."
3,IA_70,GID_IA_70_0_3,"MULTIPOLYGON Z (((8.86418 4.92709 0.00000, 8.8..."
4,IA_70,GID_IA_70_1_0,"POLYGON Z ((8.87152 5.00469 0.00000, 8.87234 5..."


In [ ]:
#gdf_mask_full.explore()

In [ ]:
# Select the road elements
xSelect = gdf_mask_full.loc[gdf_mask_full['AoI_id'].notnull(),["AoI_id","GID"]]
xSelect.head(5)


,AoI_id,GID
0,IA_70,GID_IA_70_0_0
1,IA_70,GID_IA_70_0_1
2,IA_70,GID_IA_70_0_2
3,IA_70,GID_IA_70_0_3
4,IA_70,GID_IA_70_1_0


In [ ]:
# The result of the merge with inner is the first step in creating the vector mask
gdf_mask_select= gdf_mask_full.merge(xSelect, on='GID', how='inner', suffixes=('_1', '_2'))

gdf_mask_select[['mask']] = gdf_mask_select[['AoI_id_1']] \
  .where(gdf_mask_select[['AoI_id_1']].isnull(), 1) \
  .fillna(0).astype(int)
gdf_mask_select.drop(['AoI_id_1'], axis=1, inplace=True)

gdf_mask_select = gdf_mask_select.loc[gdf_mask_select['GID'].notnull()]
print(len(gdf_mask_select))
gdf_mask_select.tail(10)


24


/var/folders/hf/1s_l6dt91218_yqmlwxj9pxm0000gn/T/ipykernel_84333/3110764797.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0).astype(int)


,GID,geometry,AoI_id_2,mask
14,GID_IA_70_0_2,"POLYGON Z ((8.84397 4.94458 0.00000, 8.84354 4...",IA_70,0
15,GID_IA_70_0_3,"POLYGON Z ((8.86581 4.94354 0.00000, 8.86565 4...",IA_70,0
16,GID_IA_70_1_0,"POLYGON Z ((8.86579 5.01323 0.00000, 8.88889 5...",IA_70,0
17,GID_IA_70_1_1,"POLYGON Z ((8.86919 4.98177 0.00000, 8.86981 4...",IA_70,0
18,GID_IA_70_1_2,"POLYGON Z ((8.88889 4.96564 0.00000, 8.87978 4...",IA_70,0
19,GID_IA_70_1_3,"MULTIPOLYGON Z (((8.86688 4.94459 0.00000, 8.8...",IA_70,0
20,GID_IA_70_2_0,"POLYGON Z ((8.88889 5.01366 0.00000, 8.91060 5...",IA_70,0
21,GID_IA_70_2_1,"MULTIPOLYGON Z (((8.91103 4.99091 0.00000, 8.9...",IA_70,0
22,GID_IA_70_2_2,"MULTIPOLYGON Z (((8.88890 4.94643 0.00000, 8.8...",IA_70,0
23,GID_IA_70_2_3,"MULTIPOLYGON Z (((8.88890 4.92289 0.00000, 8.8...",IA_70,0


In [ ]:
gdf_mask_select['mask'].value_counts()

mask
1    12
0    12
Name: count, dtype: int64

In [ ]:
working_folder = "02_WorkingFiles"

file_name = f"mask_{AoI_id}.geojson" # might be different for other areas

file_path = os.path.join(project_root, working_folder, file_name)
gdf_mask_select.to_file(file_path)

In [ ]:
# The last step is to create the ImagePatch_AoI, and this is the unmodified
# grid polygon, which is the the GID where mask = 1

# selecting rows based on condition
rslt_df = gdf_mask_select.loc[gdf_mask_select['mask'] == 1, ["GID"]]
rslt_df.head(5)

,GID
0,GID_IA_70_0_0
1,GID_IA_70_0_1
2,GID_IA_70_0_2
3,GID_IA_70_0_3
4,GID_IA_70_1_0


In [ ]:
# The result of the merge with inner is the first step in creating the vector mask
gdf_grid_select= gdf_grid.merge(rslt_df, on='GID', how='inner', suffixes=('_1', '_2'))
gdf_grid_select.head(5)

,geometry,GID
0,"POLYGON ((8.86580 4.99090, 8.84311 4.99090, 8....",GID_IA_70_0_0
1,"POLYGON ((8.86580 4.99090, 8.86580 4.96774, 8....",GID_IA_70_0_1
2,"POLYGON ((8.86580 4.96774, 8.86581 4.94458, 8....",GID_IA_70_0_2
3,"POLYGON ((8.86581 4.94458, 8.86581 4.92247, 8....",GID_IA_70_0_3
4,"POLYGON ((8.88889 4.99091, 8.86580 4.99090, 8....",GID_IA_70_1_0


In [ ]:
# define working folder
working_folder = "02_WorkingFiles"
file_name = f"ImagePatch_{AoI_id}.geojson" # might be different for other areas

# export image patch
file_path = os.path.join(project_root, working_folder, file_name)
gdf_grid_select.to_file(file_path)

In [ ]:
gdf_grid_select.explore()